# Google Colab edition

Run the setup cell first, then work from top to bottom. Keep the course ZIP in My Drive. Use a fresh Colab runtime when switching dependency profiles. **Select a GPU runtime before running this notebook.** 

Adapted from your existing exercise. Previous local outputs were cleared so new Colab runs provide their own evidence. 

Source: Original local package: 03_fastai/06_fastai_framework_and_ensemble.ipynb

In [ ]:
# Colab setup. Run this cell before the exercise cells.
from pathlib import Path
import os, sys, zipfile

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
QUICK_RUN = False  #@param {type:"boolean"}
ARCHIVE_PATH = "/content/drive/MyDrive/HaiLong_Colab_Exercises_2026-10-03.zip"  #@param {type:"string"}

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        if not Path('/content/drive/MyDrive').is_dir():
            drive.mount('/content/drive', timeout_ms=300000)
    COURSE_ROOT = Path('/content/HaiLong_Colab_Exercises')
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        archive = Path(ARCHIVE_PATH)
        if not archive.is_file():
            print('Select HaiLong_Colab_Exercises_2026-10-03.zip from your computer.')
            uploaded = files.upload()
            choices = [Path(n) for n in uploaded if n.endswith('.zip')]
            if len(choices) != 1:
                raise ValueError('Upload exactly one course ZIP file.')
            archive = choices[0]
        with zipfile.ZipFile(archive) as z:
            base = Path('/content').resolve()
            for member in z.infolist():
                target = (base / member.filename).resolve()
                if not target.is_relative_to(base) or not member.filename.startswith('HaiLong_Colab_Exercises/'):
                    raise ValueError('This is not the expected course archive.')
            z.extractall(base)
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        raise FileNotFoundError('The ZIP must contain HaiLong_Colab_Exercises/colab_runtime.py.')
else:
    # Also permits local checks without assuming the current notebook directory.
    COURSE_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'colab_runtime.py').is_file()), None)
    if COURSE_ROOT is None:
        raise RuntimeError('Open this notebook in Colab or run it inside the extracted course folder.')

os.environ['COURSE_SMOKE'] = '1' if QUICK_RUN else '0'
os.environ['WANDB_MODE'] = 'offline'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, str(COURSE_ROOT))
from colab_runtime import setup_course, finish_run
# Filled individually for each notebook by the builder.

COLAB_CONTEXT = setup_course(COURSE_ROOT, '03_fastai', 'fastai', '06_fastai_framework_and_ensemble.ipynb', USE_GOOGLE_DRIVE, install=IN_COLAB)

## Full exercise evidence

- Run all exercise cells top to bottom in hosted Colab and preserve their outputs, errors and execution counts.
- Keep QUICK_RUN=False; shortened checks or synthetic replacements for real datasets do not count as full original exercises.
- Preserve plots, models, metrics and run configurations in Drive; a final-cell marker alone is insufficient.

This cell describes evidence to collect; it does not certify completion. GPU batch caps preserve dataset coverage and epoch counts, but are documented configuration changes.


# Lesson 5: framework training and ensembling
Source task: https://github.com/fastai/course22/blob/master/06-why-you-should-use-a-framework.ipynb
This is an independently written solution companion, not a copy of the instructor notebook or a course completion record. The locally verified path uses the public Titanic teaching copy. Name/Ticket-derived features require the original Kaggle files; they are included when those columns exist.


In [ ]:
import os,json
from pathlib import Path
import numpy as np,pandas as pd,torch
import matplotlib.pyplot as plt
torch.manual_seed(42);torch.set_num_threads(1)
np.random.seed(42)
SMOKE=os.environ.get('COURSE_SMOKE','0')=='1'
results=Path('results');results.mkdir(exist_ok=True)

from fastai.tabular.all import *
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,f1_score
from fastai_utils import load_titanic,add_features
frame=add_features(load_titanic())
tr,va=train_test_split(np.arange(len(frame)),test_size=.2,random_state=42,stratify=frame.Survived)
cat_names=['Sex','Pclass','Embarked'];cont_names=['Age','SibSp','Parch','LogFare','Family','Alone']
if 'Deck' in frame:cat_names.append('Deck')
if 'Title' in frame:cat_names.append('Title')
if 'Ticket' in frame:
    ticket_counts=frame.iloc[tr].Ticket.value_counts()
    frame['TicketFreq']=frame.Ticket.map(ticket_counts).fillna(0)
    cont_names.append('TicketFreq')
to=TabularPandas(frame,procs=[Categorify,FillMissing,Normalize],cat_names=cat_names,
    cont_names=cont_names,y_names='Survived',y_block=CategoryBlock(),splits=(list(tr),list(va)))
dls=to.dataloaders(bs=64,num_workers=0)
predictions=[];runs=[]
for seed in [42,43,44]:
    set_seed(seed)
    learn=tabular_learner(dls,path=Path.cwd(),model_dir='models',layers=[20,10],metrics=accuracy)
    with learn.no_bar(),learn.no_logging():learn.fit(5 if SMOKE else 16,lr=.01)
    p,truth=learn.get_preds(dl=dls.valid);truth=truth.reshape(-1);predictions.append(p)
    assert p.shape[0] == truth.numel()
    runs.append({'seed':seed,'accuracy':float((p.argmax(1)==truth).float().mean())})
    learn.save(f'titanic_seed_{seed}')
ensemble=torch.stack(predictions).mean(0)
report={'validation_accuracy':float((ensemble.argmax(1)==truth).float().mean()),
    'validation_macro_f1':f1_score(truth.reshape(-1).numpy(),ensemble.argmax(1).numpy(),average='macro'),
    'models':3,'epochs_per_model':5 if SMOKE else 16}
print(pd.DataFrame(runs).to_string(index=False));print(report)
(results/'titanic_ensemble_metrics.json').write_text(json.dumps(report,indent=2))
pd.DataFrame({'source_row':va,'probability':ensemble[:,1].numpy()}).to_csv(results/'titanic_ensemble_validation.csv',index=False)
assert torch.isfinite(ensemble).all()

Save your outputs

Run the following cell after finishing. Choose **File > Save** in Colab to preserve your code and displayed results. Files in `results/` and `models/` persist in Drive when enabled.

In [ ]:
finish_run(COLAB_CONTEXT)